# Lab 3 — Cosmos Transfer 2.5: Photorealistic Scene Generation

Generate photorealistic training scenes from Isaac Lab sim footage using NVIDIA Cosmos Transfer 2.5-2B.

**See `README.md`** for prerequisites, NGC key setup, and quota requirements before running.

---

## Validated Status (2026-07-03)

| Component | Status |
|-----------|--------|
| EC2 Spot p5 launch (us-east-2a) | Validated |
| DLAMI bootstrap (no driver install needed) | Validated |
| NIM health endpoint | `{"status":"ready"}` |
| Inference `/v1/infer` (5 steps, 256 res) | **47s end-to-end** |
| Full restyle (35 steps, 480 res) | Not yet timed — should work |

## Prerequisites

1. **NGC API key** — must be an `nvapi-...` Personal Key from https://org.ngc.nvidia.com/setup/personal-keys with **NGC Catalog** selected. Store in Secrets Manager as `ngc-api-key`.
2. **P5 quota** — `p5.48xlarge` Spot in us-east-2. Check at Service Quotas console.
3. **Cost** — ~$13.58/hr Spot (us-east-2a). Bootstrap takes ~23 min. **Always terminate when done.**
4. **Lab 3 is optional** — Lab 4 works without Cosmos using Isaac Lab built-in domain randomization.


## 0 — Setup

In [ ]:
import boto3, importlib.util, json, os, sys, time, base64
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "aws-physical-ai-toolchain" and REPO_ROOT != REPO_ROOT.parent:
 REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Cosmos runs in us-east-2 — cheapest p5 Spot (~$13.58/hr in us-east-2a)
COSMOS_REGION = "us-east-2"
HOME_REGION = boto3.session.Session().region_name or "us-west-2"
ACCOUNT = boto3.client("sts", region_name=HOME_REGION).get_caller_identity()["Account"]
BUCKET = f"sagemaker-{HOME_REGION}-{ACCOUNT}" # default SageMaker bucket

ec2 = boto3.client("ec2", region_name=COSMOS_REGION)
ssm = boto3.client("ssm", region_name=COSMOS_REGION)

# Load cosmos_setup.py — all launch/status/generate/terminate logic lives here
cs_path = REPO_ROOT / "training" / "scripts" / "cosmos_setup.py"
spec = importlib.util.spec_from_file_location("cosmos_setup", cs_path)
cosmos_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cosmos_setup)

print(f"Account: {ACCOUNT}")
print(f"Cosmos region: {COSMOS_REGION} | Output bucket: {BUCKET}")
print(f"cosmos_setup.py loaded from {cs_path}")


## 1 — Check Prerequisites

In [ ]:
print("=" * 55)
print("Prerequisite check")
print("=" * 55)

# 1. NGC key — must be nvapi-... personal key with NGC Catalog scope
for region in [HOME_REGION, COSMOS_REGION]:
 sm = boto3.client("secretsmanager", region_name=region)
 try:
 sm.describe_secret(SecretId="ngc-api-key")
 val = sm.get_secret_value(SecretId="ngc-api-key")["SecretString"]
 prefix = val[:8] if len(val) > 8 else val
 key_type = "nvapi-... (Personal Key)" if val.startswith("nvapi-") else "legacy key (may not have NIM access)"
 print(f" OK ngc-api-key ({region}): {prefix}... [{key_type}]")
 except Exception as e:
 print(f" MISSING ngc-api-key in {region}: {e}")
 print(f" Get key: https://org.ngc.nvidia.com/setup/personal-keys")
 print(f" Store: aws secretsmanager create-secret --name ngc-api-key --secret-string 'nvapi-...' --region {region}")

# 2. IAM instance profile
iam = boto3.client("iam")
try:
 iam.get_instance_profile(InstanceProfileName="physical-ai-dev-cosmos-profile")
 print(f" OK IAM profile: physical-ai-dev-cosmos-profile")
except:
 print(f" MISSING IAM profile: physical-ai-dev-cosmos-profile")
 print(f" Run: python training/scripts/cosmos_setup.py launch --dry-run (it will error with instructions)")

# 3. Bootstrap script
ud = REPO_ROOT / "scripts" / "cosmos-userdata.sh"
print(f" {'OK' if ud.exists() else 'MISSING'} scripts/cosmos-userdata.sh ({ud.stat().st_size // 1024}KB)" if ud.exists() else f" MISSING scripts/cosmos-userdata.sh")

# 4. p5 Spot price check
try:
 hist = ec2.describe_spot_price_history(InstanceTypes=["p5.48xlarge"], ProductDescriptions=["Linux/UNIX"], MaxResults=1)["SpotPriceHistory"]
 if hist:
 print(f" OK p5.48xlarge Spot available: ${hist[0]['SpotPrice']}/hr ({hist[0]['AvailabilityZone']})")
 else:
 print(f" WARN No p5 Spot price history in {COSMOS_REGION} — may need quota increase")
except Exception as e:
 print(f" WARN Spot price check failed: {e}")

print()
print("If all OK, proceed to Section 2.")


## 2 — Launch Spot p5 Instance

Launches `p5.48xlarge` Spot in us-east-2a (8× H100, ~$13.58/hr).

**What happens automatically:**
- DLAMI boots with NVIDIA driver 570+ pre-installed (no driver build needed)
- SSM agent starts (used for all remote commands — no SSH/inbound ports)
- NGC key fetched from Secrets Manager
- `nvcr.io/nim/nvidia/cosmos-transfer2.5-2b:1.0.0` pulled (~30GB, ~15 min)
- NIM starts with H100 fp8 latency profile (CP=8, all 8 H100s)
- Model weights download and TRT engine build (~8 min)

**Total bootstrap: ~23 minutes.**


In [ ]:
# Dry run first — shows exactly what will launch, no AWS calls
import os
os.environ["COSMOS_REGION"] = COSMOS_REGION
os.environ["COSMOS_SPOT_AZ"] = "us-east-2a"

print("=== Dry run (no AWS calls) ===")
cosmos_setup.launch(dry_run=True)


In [ ]:
LAUNCH = False # ← change to True when ready

if LAUNCH:
 cosmos_setup.launch(dry_run=False)
 print()
 print("Set INSTANCE_ID in Section 3 with the ID printed above.")
 print("Bootstrap takes ~23 min. Cost: ~$13.58/hr — terminate when done.")
else:
 print("Set LAUNCH = True to launch.")
 print("Cost: ~$13.58/hr Spot (us-east-2a). ALWAYS terminate when done (Section 7).")


## 3 — Wait for NIM Ready (~23 min)

Re-run this cell every 2-3 minutes until you see `{"status":"ready"}`.

Or use `--wait` flag: `python training/scripts/cosmos_setup.py status --instance-id i-xxxx --wait`


In [ ]:
INSTANCE_ID = "" # ← paste from launch output, e.g. "i-0abc123def456"

if not INSTANCE_ID:
 print("Paste the instance ID from the launch cell above.")
else:
 inst = ec2.describe_instances(InstanceIds=[INSTANCE_ID])["Reservations"][0]["Instances"][0]
 state = inst["State"]["Name"]
 az = inst["Placement"]["AvailabilityZone"]
 print(f"Instance: {INSTANCE_ID} State: {state} AZ: {az}")

 if state == "running":
 resp = ssm.send_command(
 InstanceIds=[INSTANCE_ID],
 DocumentName="AWS-RunShellScript",
 Parameters={"commands": [
 "curl -sf http://localhost:8000/v1/health/ready 2>/dev/null || echo NOT_READY_YET; "
 "echo '---'; docker ps --format '{{.Names}} {{.Status}}' 2>/dev/null; "
 "echo '---'; docker logs cosmos 2>&1 | grep -iE 'ready|downloading|error|serving' | tail -3 2>/dev/null || true"
 ]},
 TimeoutSeconds=30,
 )
 time.sleep(12)
 out = ssm.get_command_invocation(
 CommandId=resp["Command"]["CommandId"], InstanceId=INSTANCE_ID
 ).get("StandardOutputContent", "").strip()
 print(f"\nStatus:\n{out}")
 if '"ready"' in out.lower() or '"status":' in out.lower():
 print("\n NIM is ready — proceed to Section 5.")
 else:
 print("\nNot ready yet — re-run in 2-3 min.")
 else:
 print(f"Instance is {state} — wait and retry.")


## 4 — Prepare Input Video

Cosmos Transfer requires **MP4 input, 93–480 frames**.

**Option A — Lab 1 real robot videos (available now):** 
The 27 wrist-cam demos from Lab 1 are at `training/data/ur3_lerobot_dataset/videos/`. 
They're 81–192 frames at 640×480 — Cosmos will restyle them with different environments. 
Note: real-world videos with motion blur/noise give Cosmos less structure to work with vs clean sim.

**Option B — Isaac Sim renders (recommended for production):** 
Render the same task in Isaac Lab (Lab 2 workstation) → clean, structured MP4s. 
Cosmosʼs edge/depth control works much better on clean sim geometry.

**For testing:** the cell below creates a synthetic 100-frame clip (no robot needed).


In [ ]:
# ── Option A: Use Lab 1 wrist-camera videos ───────────────────────────────────
# Lab 1 produced 27 real UR3 pick-and-place demos at 640x480, 5fps, 81-192 frames.
# Cosmos Transfer will restyle them with different lighting/surfaces.

LAB1_VIDEOS = REPO_ROOT / "training" / "data" / "ur3_lerobot_dataset" / "videos" / "chunk-000" / "observation.images.wrist"

if LAB1_VIDEOS.exists():
 lab1_clips = sorted(LAB1_VIDEOS.glob("*.mp4"))
 print(f"Lab 1 videos available: {len(lab1_clips)}")
 for v in lab1_clips[:5]:
 import cv2
 cap = cv2.VideoCapture(str(v))
 frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
 w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
 h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
 cap.release()
 print(f" {v.name}: {frames} frames, {w}x{h}, {v.stat().st_size // 1024}KB")
 print()
 # Cosmos needs 93-480 frames. Lab 1 episodes are 81-192 frames.
 # Episodes >= 93 frames can be used directly:
 valid_clips = [v for v in lab1_clips if cv2.VideoCapture(str(v)).get(cv2.CAP_PROP_FRAME_COUNT) >= 93]
 print(f"Episodes with >= 93 frames (directly usable): {len(valid_clips)}")
 print()
 print("To use: set SIM_CLIPS_DIR = LAB1_VIDEOS below, or copy specific clips.")
else:
 print("Lab 1 dataset not found. Run Lab1_Training_Inference.ipynb Section 1 first.")
 print(f"Expected path: {LAB1_VIDEOS}")


In [ ]:
import cv2, numpy as np

# ── Choose your input source ──────────────────────────────────────────────────
# Option A: Lab 1 real robot videos (set USE_LAB1 = True)
# Option B: Synthetic test clip (USE_LAB1 = False)
USE_LAB1 = False # ← set True to use real UR3 wrist-cam footage

SIM_CLIPS_DIR = Path("/tmp/cosmos_sim_clips")
SIM_CLIPS_DIR.mkdir(exist_ok=True)

if USE_LAB1 and LAB1_VIDEOS.exists():
 import shutil
 # Copy Lab 1 episodes with >= 93 frames into SIM_CLIPS_DIR
 copied = 0
 for v in sorted(LAB1_VIDEOS.glob("*.mp4")):
 cap = cv2.VideoCapture(str(v))
 frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
 cap.release()
 if frames >= 93:
 shutil.copy(v, SIM_CLIPS_DIR / v.name)
 copied += 1
 print(f"Copied {copied} Lab 1 clips (>= 93 frames) to {SIM_CLIPS_DIR}")
else:
 # Synthetic test clip — 100 frames, simple moving-arm scene
 clip_path = SIM_CLIPS_DIR / "test_scene_001.mp4"
 h, w, n_frames = 256, 256, 100
 fourcc = cv2.VideoWriter_fourcc(*"mp4v")
 writer = cv2.VideoWriter(str(clip_path), fourcc, 5, (w, h))
 for i in range(n_frames):
 frame = np.zeros((h, w, 3), dtype=np.uint8)
 frame[:h//2, :, 0] = 60 + i
 frame[h//2:, :, 1] = 40
 x = int((w - 40) * (i / n_frames))
 cv2.rectangle(frame, (x, 20), (x + 30, h - 20), (200, 200, 200), -1)
 cv2.rectangle(frame, (w//2-15, h-50), (w//2+15, h-20), (100, 150, 200), -1)
 writer.write(frame)
 writer.release()
 print(f"Created synthetic test clip: {clip_path} ({n_frames} frames)")

clips = sorted(SIM_CLIPS_DIR.glob("*.mp4"))
print(f"\nInput clips ready: {len(clips)}")
for c in clips[:5]:
 cap = cv2.VideoCapture(str(c))
 frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
 w2 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
 h2 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
 cap.release()
 print(f" {c.name}: {frames} frames, {w2}x{h2}, {c.stat().st_size//1024}KB")


## 5 — Run Cosmos Transfer

Sends the input clip to `/v1/infer` on the EC2 instance via SSM.
No inbound ports needed — everything goes through SSM.

**Validated timings (2026-07-03):**
- 5 steps, 256 res, 100 frames: **47 seconds**
- 35 steps, 480 res (production): estimated ~5-10 min


In [ ]:
COSMOS_PROMPT = "industrial warehouse with fluorescent lighting, scratched metal surfaces"
CONTROL = "edge" # edge | depth | seg | vis
NUM_STEPS = 5 # 5=fast test, 35=production quality
RESOLUTION = "256" # "256" for test, "480" for production
OUTPUT_DIR = Path("/tmp/cosmos_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Cosmos Transfer settings:")
print(f" Prompt: {COSMOS_PROMPT}")
print(f" Control: {CONTROL} (preserves edges/structure of the sim scene)")
print(f" Steps: {NUM_STEPS} (increase to 35 for production quality)")
print(f" Resolution: {RESOLUTION} (use '480' for production)")
print()

if not INSTANCE_ID:
 print("Set INSTANCE_ID in Section 3 first.")
else:
 os.environ["COSMOS_PROMPT"] = COSMOS_PROMPT
 cosmos_setup.generate(
 instance_id=INSTANCE_ID,
 input_dir=str(SIM_CLIPS_DIR),
 output_dir=str(OUTPUT_DIR),
 control=CONTROL,
 num_steps=NUM_STEPS,
 guidance=3,
 resolution=RESOLUTION,
 dry_run=False,
 )


## 6 — Download & View Output

The output MP4 is saved locally. Preview it and upload to S3 for Lab 4.


In [ ]:
from IPython.display import Video, display
import glob

output_clips = sorted(Path(OUTPUT_DIR).glob("*.mp4"))
print(f"Output clips: {len(output_clips)}")

for clip in output_clips:
 size_kb = clip.stat().st_size // 1024
 print(f" {clip.name} ({size_kb} KB)")

if output_clips:
 print(f"\nPreviewing: {output_clips[-1].name}")
 display(Video(str(output_clips[-1]), embed=True, width=512))
else:
 print("No output clips yet — run Section 5 first.")


In [ ]:
# Upload to S3 for Lab 4 RL training
if output_clips:
 s3 = boto3.client("s3", region_name=HOME_REGION)
 s3_prefix = f"cosmos-output/{INSTANCE_ID or 'test'}/"
 for clip in output_clips:
 key = s3_prefix + clip.name
 s3.upload_file(str(clip), BUCKET, key)
 print(f"Uploaded: s3://{BUCKET}/{key}")
 print(f"\nAll clips at: s3://{BUCKET}/{s3_prefix}")
else:
 print("No clips to upload.")


## 7 — Terminate Instance

**Always run this when done. ~$13.58/hr Spot.**


In [ ]:
TERMINATE = False # ← change to True when done with Cosmos

if not INSTANCE_ID:
 print("No INSTANCE_ID set.")
elif TERMINATE:
 cosmos_setup.terminate(INSTANCE_ID)
 print(f" Terminated {INSTANCE_ID}. No more charges.")
else:
 print(f"Set TERMINATE = True to stop the instance.")
 print(f"Instance to terminate: {INSTANCE_ID}")
 print(f"Cost if left running: ~$13.58/hr")


## 8 — Summary

### What you've done
- Launched Spot p5.48xlarge (8× H100) in us-east-2a
- Ran NVIDIA Cosmos Transfer 2.5-2B NIM — restyled sim clips into photorealistic video
- Output saved to `/tmp/cosmos_output/` and uploaded to `s3://{BUCKET}/cosmos-output/`

### Use in Lab 4
Cosmos output can feed Lab 4's RL training as domain randomization backgrounds.
But Lab 4 works without Cosmos using Isaac Lab's built-in randomization —
add Cosmos only if sim-to-real transfer fails on real hardware due to visual domain gap.

### Validated settings (2026-07-03)
- **Fast test:** `num_steps=5`, `resolution="256"` → **47 seconds** per 100-frame clip
- **Production:** `num_steps=35`, `resolution="480"` → estimated 5-10 min per clip
- **NGC key type:** Must be `nvapi-...` Personal Key from https://org.ngc.nvidia.com/setup/personal-keys
- **Bootstrap time:** ~23 min (DLAMI: NGC pull 15m + model load 8m)

### Reference
- `training/scripts/cosmos_setup.py` — launch/status/generate/terminate CLI
- `scripts/cosmos-userdata.sh` — EC2 bootstrap script
- `docs/cosmos-deployment-guide.md` — full runbook
